# Baselines

Two reference points, both fixed before any tuned model is built.

**Altman Z' (1983)** is the domain baseline: five ratios, weights published four decades ago,
nothing learned from this data. It answers whether a model needs to be fitted at all.

**Logistic regression** is the technical baseline: median imputation, standard scaling, balanced
class weights, all 64 features, no feature selection and none of the preprocessing planned for the
linear branch. It answers whether gradient boosting is worth reaching for.

Random ranking scores 0.070 on PR-AUC — the positive rate. Every number below is read against it.

In [1]:
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from src.baselines import AltmanZScore
from src.config import DATA_5YEAR_PATH
from src.data import split_holdout
from src.train import cross_validate_model

X_train, X_holdout, y_train, y_holdout = split_holdout(DATA_5YEAR_PATH)

Both estimators need an imputer: Altman's formula propagates NaN, and logistic regression rejects
it outright. Logistic regression additionally needs scaling — feature magnitudes here span five
orders — and `class_weight="balanced"`, without which a 7% positive rate collapses it towards a
constant. The imputer for Altman keeps pandas output because the estimator selects its five columns
by name.

In [2]:
altman_pipe = Pipeline(
    [
        ("imputer", SimpleImputer(strategy="median").set_output(transform="pandas")),
        ("model", AltmanZScore()),
    ]
)

logreg_pipe = Pipeline(
    [
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=1000, class_weight="balanced")),
    ]
)

In [3]:
models = {
    "Altman Z'": altman_pipe,
    "LogReg": logreg_pipe,
}

results = {
    name: cross_validate_model(pipe, X_train, y_train) for name, pipe in models.items()
}

table = pd.DataFrame(
    {
        name: {
            "PR-AUC": f"{r['pr_auc'].mean():.3f} ± {r['pr_auc'].std():.3f}",
            "P@top-3%": f"{r['precision_at_k'].mean():.3f} ± {r['precision_at_k'].std():.3f}",
        }
        for name, r in results.items()
    }
).T
table

,PR-AUC,P@top-3%
Altman Z',0.279 ± 0.049,0.442 ± 0.099
LogReg,0.348 ± 0.039,0.492 ± 0.081


## Reading the numbers

Altman Z' reaches 0.279 PR-AUC, a 4x lift over random ranking. Logistic regression reaches 0.348,
just under 5x. The gap between them is 0.069 against a combined spread of 0.088 — below the
threshold fixed in the evaluation protocol, so it is not read as an improvement.

The conditions behind that comparison were not equal. Altman saw none of this data: the weights
were calibrated on US manufacturing firms in the 1970s and carried over unchanged. Logistic
regression was fitted on 4387 Polish companies with 64 features instead of five. One of Altman's
five components, `Attr9` (asset turnover, weight 0.998), carries almost no separating power on this
data — its per-feature strength is 0.031 — so roughly a fifth of the formula is noise here. It
still holds.

In queue terms: of the 26 companies a fold sends to review, Altman puts about 11 real bankruptcies
in the list and logistic regression about 12, against roughly 2 under random ordering.

The bar for everything that follows is 0.279, and an improvement has to clear it by more than the
summed spreads.